# Minimal DS Agent Notebook for Prompt Experimentation

A standalone notebook that replicates the core DS Agent workflow (code generation + execution) for prompt tuning.

**Features:**
- Standalone (no backend API dependencies)
- All prompts editable in one place
- Supports Claude and OpenAI
- Output format compatible with eval_GPT_enhanced
- PNG plots for publication quality

**Workflow:**
```
Excel File → Extract Metadata → Build Prompt → LLM (Code Gen) → Execute Python → Capture Stdout → Save as "response"
```

## Usage Guide

### Quick Start

1. **Configure API keys** (Cell 1):
   ```python
   Config.ANTHROPIC_API_KEY = "sk-ant-..."
   Config.OPENAI_API_KEY = "sk-..."
   Config.OPENROUTER_API_KEY = "sk-or-v1-..."
   ```

2. **Test single query** (Cell 5):
   - Update `EXCEL_FILE`, `INTRODUCTION_FILE`, `QUESTION_FILE`
   - Choose API type: `"claude"`, `"openai"`, or `"openrouter"`
   - Run cell to see results

3. **Batch evaluation** (Cell 6):
   - Load samples from `data_sample.json`
   - Run `batch_evaluate()` function
   - Results saved in eval_GPT_enhanced format

### API Options

The notebook supports three API types:

1. **Direct Anthropic API** (`api_type="claude"`)
   - Uses official Anthropic SDK
   - Requires `ANTHROPIC_API_KEY`
   - Default model: `claude-sonnet-4-20250514`

2. **Direct OpenAI API** (`api_type="openai"`)
   - Uses official OpenAI SDK
   - Requires `OPENAI_API_KEY`
   - Default model: `gpt-4o`

3. **OpenRouter** (`api_type="openrouter"`)
   - Unified gateway for multiple LLM providers
   - Requires `OPENROUTER_API_KEY` and explicit `model` parameter
   - Supports both Claude and GPT models:
     - `"anthropic/claude-opus-4.5"`
     - `"anthropic/claude-sonnet-4-20250514"`
     - `"openai/gpt-4o"`
     - `"openai/gpt-4-turbo"`
   - Automatically forces correct provider routing

### Prompt Experimentation

Edit `PromptTemplates.CODE_GENERATION_TEMPLATE` in Cell 2:
- Add/remove requirements
- Change temperature
- Modify instructions
- Compare different models and providers

### Output Format

Results match eval_GPT_enhanced format:
```python
{
  "id": "00000001",
  "question_idx": 0,
  "model": "dsagent-claude",
  "time": 2.5,
  "response": "Cap price is $81.97\n..."  # ← Evaluated against ground truth
}
```

### Dependencies

```bash
conda create -n olive python=3.14 -y
conda activate olive
```

Required packages:
```bash
pip install requirements.txt
```

### Advantages

- ✅ Standalone (no backend server needed)
- ✅ All prompts in one place for easy editing
- ✅ PNG plots for publication quality
- ✅ Compatible with existing evaluation scripts
- ✅ Transparent workflow for debugging
- ✅ Flexible API options (Anthropic, OpenAI, OpenRouter)
- ✅ Easy model comparison for prompt optimization

In [11]:
# ===== CELL 1: Imports and Configuration =====

import pandas as pd
import numpy as np
import json
import sys
import io
import os
import time
import re
from contextlib import redirect_stdout
from datetime import datetime
import traceback
from typing import Dict, List, Any, Optional
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv

# LLM APIs
import anthropic
from openai import OpenAI
import requests  # For OpenRouter

# Plotly for PNG plots
import plotly.express as px
import plotly.graph_objects as go
# Note: kaleido is required for PNG export
# Install with: pip install kaleido

# Load environment variables
load_dotenv()

class Config:
    """Configuration class for easy parameter adjustment"""
    
    # API Keys (set in environment or here)
    ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
    
    # OpenRouter configuration
    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
    USE_OPENROUTER = os.getenv("USE_OPENROUTER", "false").lower() == "true"
    
    # Default models
    CLAUDE_MODEL = "claude-sonnet-4-20250514"
    OPENAI_MODEL = "gpt-4o"
    
    # OpenRouter model identifiers (use these when api_type="openrouter")
    OPENROUTER_CLAUDE_OPUS = "anthropic/claude-opus-4.5"
    OPENROUTER_CLAUDE_SONNET = "anthropic/claude-sonnet-4-20250514"
    OPENROUTER_GPT4 = "openai/gpt-4o"
    OPENROUTER_GPT4_TURBO = "openai/gpt-4-turbo"
    
    # LLM Parameters
    TEMPERATURE = 0.3
    MAX_TOKENS = 4000
    
    # Output directories
    PLOTS_DIR = "outputs/notebook_plots"
    RESULTS_DIR = "outputs/notebook_results"

# Create output directories
os.makedirs(Config.PLOTS_DIR, exist_ok=True)
os.makedirs(Config.RESULTS_DIR, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Anthropic API: {'Configured' if Config.ANTHROPIC_API_KEY else 'Not configured'}")
print(f"  OpenAI API: {'Configured' if Config.OPENAI_API_KEY else 'Not configured'}")
print(f"  OpenRouter API: {'Configured' if Config.OPENROUTER_API_KEY else 'Not configured'}")
print(f"  Output directories created")

✓ Configuration loaded
  Anthropic API: Not configured
  OpenAI API: Configured
  OpenRouter API: Configured
  Output directories created


In [2]:
# ===== CELL 2: Prompt Templates =====

class PromptTemplates:
    """Centralized prompt management for easy experimentation"""
    
    # Main code generation prompt (adapted from prompts.js:144-180 SINGLE_SNIPPET)
    CODE_GENERATION_TEMPLATE = """You are a data analyst. The user asked: "{{userMessage}}"

Dataset Information:
{{dataInfo}}

1. Generate ONE COMPLETE, STANDALONE code snippet that does all the analysis in one go. Return it as a single element in the steps array.
For this step, provide:
1. A text explanation of what this step does
2. The Python code snippet that implements this step

Format your response as JSON with this structure:
{
  "steps": [
    {
      "explanation": "Detailed explanation of what this step accomplishes",
      "code": "Complete Python code with imports, analysis, and output"
    }
  ]
}

REQUIREMENTS:
- The data is already loaded and available as 'data' variable
- Use the 'data' variable directly instead of reading the file
- DO NOT include pd.read_csv() calls in your code
- MUST include at least one visualization using plotly == 6.3.0, use update_xaxes and update_yaxes for axis formatting
- Use only plotly for visualizations
- Subplots MUST have 2 graphs per row maximum
- Use update_xaxes and update_yaxes for axis formatting
- Include necessary imports in the snippet
- Complete analysis logic
- Clear visualization code when applicable
- Proper print() statements to show results
- The code should work as a standalone script
- Do not exceed 2000 characters in your response.
- Avoid deprecated pandas patterns: do not use groupby(...).apply(...) in a way that depends on grouping columns being present. Prefer groupby(...).agg(...) and dont use include_groups attribute. The code should not trigger pandas deprecation warnings.
"""
    
    @staticmethod
    def replace_variables(template: str, variables: Dict[str, str]) -> str:
        """Replace {{variable}} placeholders in template"""
        result = template
        for key, value in variables.items():
            result = result.replace(f"{{{{{key}}}}}", str(value))
        return result
    
    @staticmethod
    def get_code_generation_prompt(user_message: str, data_info: Dict) -> str:
        """Generate the full code generation prompt"""
        return PromptTemplates.replace_variables(
            PromptTemplates.CODE_GENERATION_TEMPLATE,
            {
                "userMessage": user_message,
                "dataInfo": json.dumps(data_info, indent=2)
            }
        )

print("✓ Prompt templates loaded")
print("  Modify PromptTemplates.CODE_GENERATION_TEMPLATE to experiment with prompts")

✓ Prompt templates loaded
  Modify PromptTemplates.CODE_GENERATION_TEMPLATE to experiment with prompts


In [3]:
# ===== CELL 3: Helper Functions =====

# --- 1. Extract Minimal Metadata ---

def extract_minimal_metadata(df: pd.DataFrame, filename: str) -> Dict:
    """
    Extract minimal metadata for LLM prompt.
    Focus: What helps LLM generate accurate code (no API compatibility needed)
    """
    metadata = {
        "rowCount": len(df),
        "colsCount": len(df.columns),
        "columns": []
    }
    
    # Column information
    for col in df.columns:
        metadata["columns"].append({
            "name": str(col),
            "type": str(df[col].dtype)
        })
    
    # Simple summary
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    summary_parts = [f"{filename} contains {len(df)} rows and {len(df.columns)} columns."]
    if numeric_cols:
        cols_str = ', '.join(numeric_cols[:5])
        if len(numeric_cols) > 5:
            cols_str += '...'
        summary_parts.append(f"Numeric columns: {cols_str}.")
    if categorical_cols:
        cols_str = ', '.join(categorical_cols[:5])
        if len(categorical_cols) > 5:
            cols_str += '...'
        summary_parts.append(f"Categorical columns: {cols_str}.")
    
    metadata["summary"] = " ".join(summary_parts)
    
    return metadata


# --- 2. Call LLM API ---

def call_llm_for_code(prompt: str, api_type: str = "claude", **kwargs) -> Dict:
    """
    Call Claude, OpenAI, or OpenRouter to generate Python code.
    Returns {success, code, explanation, error}
    
    Args:
        prompt: The full prompt for code generation
        api_type: "claude", "openai", or "openrouter"
        **kwargs: Additional parameters:
            - temperature: Temperature for sampling (default from Config)
            - max_tokens: Max tokens to generate (default from Config)
            - model: Model to use (required for openrouter)
    """
    temperature = kwargs.get("temperature", Config.TEMPERATURE)
    max_tokens = kwargs.get("max_tokens", Config.MAX_TOKENS)
    
    try:
        if api_type.lower() == "claude":
            if not Config.ANTHROPIC_API_KEY:
                return {"success": False, "error": "Anthropic API key not configured"}
            
            client = anthropic.Anthropic(api_key=Config.ANTHROPIC_API_KEY)
            response = client.messages.create(
                model=kwargs.get("model", Config.CLAUDE_MODEL),
                max_tokens=max_tokens,
                temperature=temperature,
                system="You are a data analyst. Generate Python code to analyze data.",
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text
            
        elif api_type.lower() == "openai":
            if not Config.OPENAI_API_KEY:
                return {"success": False, "error": "OpenAI API key not configured"}
            
            client = OpenAI(api_key=Config.OPENAI_API_KEY)
            response = client.chat.completions.create(
                model=kwargs.get("model", Config.OPENAI_MODEL),
                max_tokens=max_tokens,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": "You are a data analyst. Generate Python code."},
                    {"role": "user", "content": prompt}
                ]
            )
            response_text = response.choices[0].message.content
            
        elif api_type.lower() == "openrouter":
            if not Config.OPENROUTER_API_KEY:
                return {"success": False, "error": "OpenRouter API key not configured"}
            
            # Model is required for OpenRouter
            model = kwargs.get("model")
            if not model:
                return {"success": False, "error": "Model must be specified for OpenRouter (e.g., 'anthropic/claude-opus-4.5')"}
            
            # Build request payload
            payload = {
                "model": model,
                "messages": [
                    {"role": "system", "content": "You are a data analyst. Generate Python code to analyze data."},
                    {"role": "user", "content": prompt}
                ],
                "temperature": temperature,
                "max_tokens": max_tokens
            }
            
            # Add provider forcing if model contains provider prefix
            # This ensures we use the specified provider (e.g., force Anthropic for anthropic/* models)
            if "/" in model:
                provider = model.split("/")[0]
                payload["provider"] = {
                    "order": [provider]
                }
            
            # Make request to OpenRouter
            headers = {
                "Authorization": f"Bearer {Config.OPENROUTER_API_KEY}",
                "Content-Type": "application/json"
            }
            
            response = requests.post(
                Config.OPENROUTER_BASE_URL,
                headers=headers,
                json=payload,
                timeout=60
            )
            
            if response.status_code != 200:
                return {
                    "success": False,
                    "error": f"OpenRouter API error: {response.status_code} - {response.text}"
                }
            
            response_data = response.json()
            response_text = response_data["choices"][0]["message"]["content"]
        
        else:
            return {"success": False, "error": f"Unknown API type: {api_type}. Use 'claude', 'openai', or 'openrouter'"}
        
        # Parse JSON response
        cleaned = response_text.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
        
        parsed = json.loads(cleaned)
        
        if "steps" in parsed and len(parsed["steps"]) > 0:
            step = parsed["steps"][0]
            return {
                "success": True,
                "code": step.get("code", ""),
                "explanation": step.get("explanation", ""),
                "raw_response": response_text
            }
        else:
            return {"success": False, "error": "No steps found in response"}
    
    except json.JSONDecodeError as e:
        return {
            "success": False,
            "error": f"JSON parse error: {str(e)}",
            "raw_response": response_text if 'response_text' in locals() else None
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "traceback": traceback.format_exc()
        }


# --- 3. Enhance Code for PNG Plots ---

def enhance_code_for_png(code: str, session_id: str) -> str:
    """
    Replace fig.show() with fig.write_image() to save PNG plots.
    """
    lines = code.split("\n")
    enhanced = []
    plot_counter = 1
    
    for line in lines:
        # Skip pd.read_csv() - data already loaded
        if "pd.read_csv(" in line or "pd.read_excel(" in line:
            continue
        
        # Replace fig.show() with PNG saving
        if ".show()" in line and "fig" in line:
            fig_match = re.search(r'(\w+)\.show\(\)', line)
            if fig_match:
                fig_var = fig_match.group(1)
                indent = " " * (len(line) - len(line.lstrip()))
                
                plot_filename = f"{Config.PLOTS_DIR}/plot_{session_id}_{plot_counter}.png"
                enhanced.append(f"{indent}{fig_var}.write_image('{plot_filename}', width=800, height=600)")
                
                plot_counter += 1
                continue
        
        enhanced.append(line)
    
    return "\n".join(enhanced)


# --- 4. Execute Python Code ---

def execute_code(code: str, df: pd.DataFrame, session_id: str) -> Dict:
    """
    Execute generated code and capture stdout.
    Returns {success, output, plots, error}
    """
    # Enhance code to save PNG plots
    enhanced_code = enhance_code_for_png(code, session_id)
    
    # Execution namespace
    namespace = {
        'data': df,
        'pd': pd,
        'np': np,
        'px': px,
        'go': go,
    }
    
    # Capture stdout
    output_buffer = io.StringIO()
    plots = []
    
    try:
        with redirect_stdout(output_buffer):
            exec(enhanced_code, namespace)
        
        # Collect generated plots
        if os.path.exists(Config.PLOTS_DIR):
            for file in os.listdir(Config.PLOTS_DIR):
                if file.startswith(f"plot_{session_id}_"):
                    plots.append(os.path.join(Config.PLOTS_DIR, file))
        
        return {
            "success": True,
            "output": output_buffer.getvalue(),  # This becomes "response" field
            "plots": plots,
            "code_executed": enhanced_code
        }
    
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "traceback": traceback.format_exc(),
            "output": output_buffer.getvalue(),
            "plots": plots
        }

print("✓ Helper functions loaded")
print("  1. extract_minimal_metadata()")
print("  2. call_llm_for_code() - Supports: claude, openai, openrouter")
print("  3. enhance_code_for_png()")
print("  4. execute_code()")

✓ Helper functions loaded
  1. extract_minimal_metadata()
  2. call_llm_for_code() - Supports: claude, openai, openrouter
  3. enhance_code_for_png()
  4. execute_code()


In [4]:
# ===== CELL 4: Main Workflow Function =====

def process_single_query(
    excel_path: str,
    introduction: str,
    question: str,
    api_type: str = "claude",
    **llm_kwargs
) -> Dict:
    """
    End-to-end processing: Excel + Query → Code Generation → Execution → Output
    
    Args:
        excel_path: Path to Excel/CSV file
        introduction: Introduction text (context)
        question: The question to answer
        api_type: "claude", "openai", or "openrouter"
        **llm_kwargs: LLM parameters (temperature, model, etc.)
    
    OpenRouter Examples:
        # Using Claude via OpenRouter
        result = process_single_query(
            excel_path="data.xlsx",
            introduction="...",
            question="...",
            api_type="openrouter",
            model="anthropic/claude-opus-4.5",
            temperature=0.3
        )
        
        # Using GPT-4 via OpenRouter
        result = process_single_query(
            excel_path="data.xlsx",
            introduction="...",
            question="...",
            api_type="openrouter",
            model="openai/gpt-4o",
            temperature=0.3
        )
    
    Returns:
        Dict matching eval_GPT_enhanced format:
        {
            "success": True/False,
            "time": 1.5,
            "response": "Output text...",  # ← Key field for evaluation
            "code": "Generated Python code",
            "explanation": "What the code does",
            "plots": ["path1.png", "path2.png"],
            "error": "Error message if failed"
        }
    """
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    start_time = time.time()
    
    result = {
        "session_id": session_id,
        "success": False,
        "timestamp": datetime.now().isoformat()
    }
    
    try:
        # Step 1: Load Excel/CSV
        print(f"[{session_id}] Loading data from {os.path.basename(excel_path)}...")
        if excel_path.endswith('.csv'):
            df = pd.read_csv(excel_path)
        else:
            df = pd.read_excel(excel_path)
        print(f"  ✓ Loaded: {len(df)} rows, {len(df.columns)} columns")
        
        # Step 2: Extract minimal metadata
        print(f"[{session_id}] Extracting metadata...")
        metadata = extract_minimal_metadata(df, os.path.basename(excel_path))
        result["metadata"] = metadata
        print(f"  ✓ Metadata extracted")
        
        # Step 3: Build prompt
        print(f"[{session_id}] Building prompt...")
        full_query = f"{introduction}\n\n{question}" if introduction else question
        prompt = PromptTemplates.get_code_generation_prompt(full_query, metadata)
        result["prompt"] = prompt
        print(f"  ✓ Prompt built ({len(prompt)} chars)")
        
        # Step 4: Call LLM for code
        api_display = api_type.upper()
        if api_type.lower() == "openrouter" and "model" in llm_kwargs:
            api_display = f"OPENROUTER ({llm_kwargs['model']})"
        print(f"[{session_id}] Calling {api_display} API...")
        
        llm_result = call_llm_for_code(prompt, api_type, **llm_kwargs)
        
        if not llm_result["success"]:
            result["error"] = llm_result["error"]
            result["time"] = time.time() - start_time
            print(f"  ✗ LLM call failed: {llm_result['error']}")
            return result
        
        result["code"] = llm_result["code"]
        result["explanation"] = llm_result["explanation"]
        print(f"  ✓ Code generated ({len(llm_result['code'])} chars)")
        
        # Step 5: Execute code
        print(f"[{session_id}] Executing code...")
        exec_result = execute_code(llm_result["code"], df, session_id)
        
        result["execution"] = exec_result
        result["success"] = exec_result["success"]
        result["time"] = time.time() - start_time
        
        # Step 6: Format for eval compatibility
        result["response"] = exec_result.get("output", "")  # ← This is evaluated
        result["plots"] = exec_result.get("plots", [])
        
        if exec_result["success"]:
            print(f"  ✓ Code executed successfully")
            print(f"  ✓ Generated {len(exec_result['plots'])} plot(s)")
        else:
            result["error"] = exec_result.get("error", "")
            print(f"  ✗ Execution failed: {exec_result.get('error', 'Unknown')}")
        
        return result
    
    except Exception as e:
        result["error"] = str(e)
        result["traceback"] = traceback.format_exc()
        result["time"] = time.time() - start_time
        print(f"  ✗ Error: {e}")
        return result

print("✓ Main workflow function loaded")
print("  Use: result = process_single_query(excel_path, introduction, question, api_type)")
print("  Supported api_type: 'claude', 'openai', 'openrouter'")

✓ Main workflow function loaded
  Use: result = process_single_query(excel_path, introduction, question, api_type)
  Supported api_type: 'claude', 'openai', 'openrouter'


In [12]:
# ===== CELL 5: Single Query Testing =====

# Configuration
EXCEL_FILE = "D:\\VSCode_Projects\\DSBench\\data_analysis\\data_sample\\00000001\\MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx"  # Change this
INTRODUCTION_FILE = "D:\\VSCode_Projects\\DSBench\\data_analysis\\data_sample\\00000001\\introduction.txt"  # Optional
QUESTION_FILE = "D:\\VSCode_Projects\\DSBench\\data_analysis\\data_sample\\00000001\\question6.txt"  # Change this

# Check if files exist
if not os.path.exists(EXCEL_FILE):
    print("❌ Excel file not found. Please update EXCEL_FILE path.")
else:
    # Load introduction
    if os.path.exists(INTRODUCTION_FILE):
        with open(INTRODUCTION_FILE, 'r', encoding='utf-8') as f:
            introduction = f.read().strip()
    else:
        introduction = ""
        print("⚠️ Introduction file not found, using empty introduction")
    
    # Load question
    if os.path.exists(QUESTION_FILE):
        with open(QUESTION_FILE, 'r', encoding='utf-8') as f:
            question = f.read().strip()
    else:
        print("❌ Question file not found. Please update QUESTION_FILE path.")
        question = None
    
    if question:
        # Process query
        print("\n" + "="*80)
        print("PROCESSING SINGLE QUERY")
        print("="*80)
        
        # ===== OPTION 1: Direct Anthropic API =====
        # result = process_single_query(
        #     excel_path=EXCEL_FILE,
        #     introduction=introduction,
        #     question=question,
        #     api_type="claude",
        #     temperature=0.3
        #     # model="claude-opus-4-20250514"  # Optional: override model
        # )
        
        # ===== OPTION 2: Direct OpenAI API =====
        # result = process_single_query(
        #     excel_path=EXCEL_FILE,
        #     introduction=introduction,
        #     question=question,
        #     api_type="openai",
        #     temperature=0.3
        #     # model="gpt-4o"  # Optional: override model
        # )
        
        # ===== OPTION 3: OpenRouter with Claude =====
        result = process_single_query(
            excel_path=EXCEL_FILE,
            introduction=introduction,
            question=question,
            api_type="openrouter",
            model="anthropic/claude-opus-4.5",  # Required for OpenRouter
            temperature=0.3
        )
        
        # ===== OPTION 4: OpenRouter with GPT-4 =====
        # result = process_single_query(
        #     excel_path=EXCEL_FILE,
        #     introduction=introduction,
        #     question=question,
        #     api_type="openrouter",
        #     model="openai/gpt-4o",  # Required for OpenRouter
        #     temperature=0.3
        # )
        
        # Display results
        print("\n" + "="*80)
        print("RESULTS")
        print("="*80)
        print(f"Success: {result['success']}")
        print(f"Time: {result.get('time', 0):.2f}s")
        
        if result.get('explanation'):
            print(f"\nExplanation:\n{result['explanation']}")
        
        if result.get('response'):
            print(f"\nResponse (for evaluation):\n{result['response']}")
        
        print(f"\nPlots generated: {len(result.get('plots', []))}")
        if result.get('plots'):
            for plot in result['plots']:
                print(f"  - {plot}")
        
        if not result['success']:
            print(f"\nError: {result.get('error', 'Unknown error')}")
            if result.get('traceback'):
                print(f"\nTraceback:\n{result['traceback']}")
        
        print("="*80)


PROCESSING SINGLE QUERY
[20260131_213228_057314] Loading data from MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  ✓ Loaded: 47 rows, 9 columns
[20260131_213228_057314] Extracting metadata...
  ✓ Metadata extracted
[20260131_213228_057314] Building prompt...
  ✓ Prompt built (7096 chars)
[20260131_213228_057314] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (1650 chars)
[20260131_213228_057314] Executing code...
  ✗ Execution failed: Trace type 'pie' is not compatible with subplot type 'xy'
at grid position (1, 1)

See the docstring for the specs argument to plotly.subplots.make_subplots
for more information on subplot types

RESULTS
Success: False
Time: 16.23s

Explanation:
Calculate operational days in Q4 2023 (Oct 1 - Dec 31). Q4 2023 has 92 days total. There's a shutdown from Oct 25 to Nov 10, 2023 (17 days). So operational days = 92 - 17 = 75 days.

Response (for evaluation):
Total days in Q4 2023: 92
Shutdown days (Oct 25 - Nov 10): 17

Operational

In [ ]:
# ===== CELL 6: Batch Evaluation (Compatible with eval_GPT_enhanced) =====

def batch_evaluate(
    samples: List[Dict],
    data_path: str,
    api_type: str = "claude",
    save_path: Optional[str] = None,
    **llm_kwargs
) -> List[Dict]:
    """
    Process multiple samples/questions.
    Output format matches eval_GPT_enhanced for accuracy computation.
    
    Args:
        samples: List from data_sample.json
        data_path: Base path to sample folders (e.g., "./data_sample/")
        api_type: "claude", "openai", or "openrouter"
        save_path: Optional folder to save results
        **llm_kwargs: LLM parameters (temperature, model, etc.)
    
    Returns:
        List of dicts matching eval_GPT_enhanced format
    """
    from tqdm.auto import tqdm
    
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        print(f"✓ Results will be saved to: {save_path}")
    
    all_results = []
    total_time = 0
    
    for sample_idx, sample in enumerate(tqdm(samples, desc="Samples")):
        sample_id = sample["id"]
        sample_dir = os.path.join(data_path, sample_id)
        
        if not os.path.exists(sample_dir):
            print(f"⚠️ Sample directory not found: {sample_dir}, skipping")
            continue
        
        # Load introduction
        intro_file = os.path.join(sample_dir, "introduction.txt")
        if os.path.exists(intro_file):
            with open(intro_file, 'r', encoding='utf-8') as f:
                introduction = f.read().strip()
        else:
            introduction = ""
        
        # Find Excel file
        excel_files = [f for f in os.listdir(sample_dir)
                      if f.lower().endswith(('.xlsx', '.xlsb', '.xlsm', '.csv'))]
        if not excel_files:
            print(f"⚠️ No Excel file found for {sample_id}, skipping")
            continue
        
        excel_path = os.path.join(sample_dir, excel_files[0])
        
        # Process each question
        for question_idx, question_name in enumerate(sample["questions"]):
            question_file = os.path.join(sample_dir, f"{question_name}.txt")
            
            if not os.path.exists(question_file):
                print(f"⚠️ Question file not found: {question_file}, skipping")
                continue
            
            with open(question_file, 'r', encoding='utf-8') as f:
                question = f.read().strip()
            
            print(f"\nProcessing {sample_id} - Question {question_idx + 1}/{len(sample['questions'])}")
            
            # Process query
            result = process_single_query(
                excel_path=excel_path,
                introduction=introduction,
                question=question,
                api_type=api_type,
                **llm_kwargs
            )
            
            # Format for eval compatibility (matching eval_GPT_enhanced)
            model_name = f"dsagent-{api_type}"
            if api_type.lower() == "openrouter" and "model" in llm_kwargs:
                model_name = f"dsagent-openrouter-{llm_kwargs['model'].replace('/', '-')}"
            
            eval_entry = {
                "id": sample_id,
                "question_idx": question_idx,
                "model": model_name,
                "input_tokens": 0,  # Not tracked for simplicity
                "output_tokens": 0,
                "cost": 0.0,  # Not tracked
                "time": result.get("time", 0),
                "response": result.get("response", "")  # ← Key field for evaluation
            }
            
            # Add error if failed
            if not result.get("success"):
                eval_entry["error"] = result.get("error", "Unknown error")
            
            all_results.append(eval_entry)
            total_time += result.get("time", 0)
            
            # Save to file (same format as eval_GPT_enhanced)
            if save_path:
                output_file = os.path.join(save_path, f"{sample_id}.json")
                with open(output_file, 'a', encoding='utf-8') as f:
                    json.dump(eval_entry, f)
                    f.write("\n")
    
    # Summary
    print("\n" + "="*80)
    print("BATCH EVALUATION SUMMARY")
    print("="*80)
    total = len(all_results)
    successful = sum(1 for r in all_results if "error" not in r)
    print(f"Total questions: {total}")
    print(f"Successful: {successful} ({successful/total*100:.1f}%)")
    print(f"Failed: {total - successful} ({(total-successful)/total*100:.1f}%)")
    print(f"Total time: {total_time:.1f}s ({total_time/60:.1f}m)")
    if successful > 0:
        print(f"Avg time per question: {total_time/total:.1f}s")
    if save_path:
        print(f"Results saved to: {save_path}")
    print("="*80)
    
    return all_results


# ===== Example Usage =====

# Load samples from data_sample.json (uncomment to run)
samples = []
with open("./data_sample.json", "r") as f:
    for line in f:
        samples.append(eval(line.strip()))

# ===== OPTION 1: Direct Anthropic Claude API =====
# results = batch_evaluate(
#     samples=samples,
#     data_path="./data_sample/",
#     api_type="claude",
#     save_path="./outputs/dsagent_minimal/claude/",
#     temperature=0.3
# )

# ===== OPTION 2: Direct OpenAI API =====
# results = batch_evaluate(
#     samples=samples,
#     data_path="./data_sample/",
#     api_type="openai",
#     save_path="./outputs/dsagent_minimal/openai/",
#     temperature=0.3
# )

# ===== OPTION 3: OpenRouter with Claude Opus 4.5 =====
results = batch_evaluate(
    samples=samples,
    data_path="./data_sample/",
    api_type="openrouter",
    model="anthropic/claude-opus-4.5",  # Required for OpenRouter
    save_path="./outputs/dsagent_minimal/openrouter_claude/",
    temperature=0.3
)

# ===== OPTION 4: OpenRouter with GPT-4o =====
# results = batch_evaluate(
#     samples=samples,
#     data_path="./data_sample/",
#     api_type="openrouter",
#     model="openai/gpt-4o",  # Required for OpenRouter
#     save_path="./outputs/dsagent_minimal/openrouter_gpt4/",
#     temperature=0.3
# )

print("✓ Batch evaluation function loaded")
print("  Use: results = batch_evaluate(samples, data_path, api_type, save_path)")
print("  Supported api_type: 'claude', 'openai', 'openrouter'")
print("  Output format matches eval_GPT_enhanced for accuracy computation")

✓ Results will be saved to: ./save_process/dsagent_minimal/openrouter_claude/



Processing 00000001 - Question 1/13
[20260131_213747_035152] Loading data from MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  ✓ Loaded: 47 rows, 9 columns
[20260131_213747_035152] Extracting metadata...
  ✓ Metadata extracted
[20260131_213747_035152] Building prompt...
  ✓ Prompt built (7096 chars)
[20260131_213747_035152] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (1782 chars)
[20260131_213747_035152] Executing code...
  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

Processing 00000001 - Question 2/13
[20260131_213801_786037] Loading data from MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  ✓ Loaded: 47 rows, 9 columns
[20260131_213801_786037] Extracting metadata...
  ✓ Metadata extracted
[20260131_213801_786037] Building prompt...
  ✓ Prompt built (7160 chars)
[20260131_213801_786037] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (2282 chars)
[20260131_213801_786037] Executing code...
  ✓ Code executed success

Exception ignored while calling deallocator <function tqdm.__del__ at 0x000001F90605BED0>:
Traceback (most recent call last):
  File "c:\Users\ustra\miniconda3\envs\olive\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\ustra\miniconda3\envs\olive\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

Processing 00000001 - Question 10/13
[20260131_214027_099349] Loading data from MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  ✓ Loaded: 47 rows, 9 columns
[20260131_214027_099349] Extracting metadata...
  ✓ Metadata extracted
[20260131_214027_099349] Building prompt...
  ✓ Prompt built (7339 chars)
[20260131_214027_099349] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (3815 chars)
[20260131_214027_099349] Executing code...
  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

Processing 00000001 - Question 11/13
[20260131_214054_710138] Loading data from MO16-Round-1-Sec-2-Chip-Off-The-Old-Block.xlsx...
  ✓ Loaded: 47 rows, 9 columns
[20260131_214054_710138] Extracting metadata...
  ✓ Metadata extracted
[20260131_214054_710138] Building prompt...
  ✓ Prompt built (7305 chars)
[20260131_214054_710138] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (3762 chars)
[20260131_2

  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

Processing 00000038 - Question 1/5
[20260131_214212_532869] Loading data from MO13-Energy-Operations-Workbook.xlsx...
  ✓ Loaded: 59 rows, 7 columns
[20260131_214212_532869] Extracting metadata...
  ✓ Metadata extracted
[20260131_214212_532869] Building prompt...
  ✓ Prompt built (6365 chars)
[20260131_214212_532869] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (2193 chars)
[20260131_214212_532869] Executing code...
  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

Processing 00000038 - Question 2/5
[20260131_214227_833160] Loading data from MO13-Energy-Operations-Workbook.xlsx...
  ✓ Loaded: 59 rows, 7 columns
[20260131_214227_833160] Extracting metadata...
  ✓ Metadata extracted
[20260131_214227_833160] Building prompt...
  ✓ Prompt built (6344 chars)
[20260131_214227_833160] Calling OPENROUTER (anthropic/claude-opus-4.5) API...
  ✓ Code generated (2761 chars)
[20260131_214227_833160] Executing 


Samples: 100%|██████████| 2/2 [05:52<00:00, 176.23s/it]

  ✓ Code executed successfully
  ✓ Generated 1 plot(s)

BATCH EVALUATION SUMMARY
Total questions: 18
Successful: 16 (88.9%)
Failed: 2 (11.1%)
Total time: 352.4s (5.9m)
Avg time per question: 19.6s
Results saved to: ./save_process/dsagent_minimal/openrouter_claude/
✓ Batch evaluation function loaded
  Use: results = batch_evaluate(samples, data_path, api_type, save_path)
  Supported api_type: 'claude', 'openai', 'openrouter'
  Output format matches eval_GPT_enhanced for accuracy computation


## Error Fix: IProgress Widget Missing

The error occurs because `tqdm.notebook` requires the `ipywidgets` package for displaying progress bars in Jupyter notebooks.

**Solution:** Update the import in **CELL INDEX: 7** to use `tqdm.auto` instead of `tqdm.notebook`:

```python
# Change this line in the batch_evaluate function:
from tqdm.auto import tqdm  # Changed from tqdm.notebook
```

**Alternative Solutions:**

1. **Install/Update ipywidgets** (if you want fancy notebook progress bars):
    ```bash
    pip install --upgrade ipywidgets
    jupyter nbextension enable --py widgetsnbextension
    ```

2. **Use standard tqdm** (simple text progress bar):
    ```python
    from tqdm import tqdm
    ```

**Recommended:** Use `tqdm.auto` as it automatically detects the environment and uses the appropriate progress bar (notebook widget if available, otherwise falls back to text-based).